# Debye-Waller workflow with a machine-learned potential

This notebook computes thermal displacement matrices, and diffraction patterns
with Debye-Waller factors, with `DebyeWallerMaker`. The forces come from
MACE-OMAT-0-medium. For the polar materials, the Born charges and the
high-frequency dielectric tensor for the non-analytical correction (NAC) come
from MACE-Field.

| Part | Content | Needs | Time |
|---|---|---|---|
| A. Run the workflows | Si step by step, six materials, a q-point scan for Cu, a reference from the Materials Project | MACE-Field, a pymatgen with anisotropic Debye-Waller factors, a Materials Project API key | 3 to 4 min per material on one A100 GPU |
| B. Our results | All figures and tables, drawn from the numbers of our runs | atomate2 | seconds |

To see the results, run only Part B. Select its first code cell and use "Run
Selected Cell and All Below". "Run All" also starts the workflow runs of Part A.

Contents:

- Background
- Installation
- Part A. Run the workflows
  - A1. Si step by step
  - A2. Six materials and a q-point scan for Cu
  - A3. Reference from the Materials Project
  - A4. Saving the numbers for Part B
- Part B. Our results
  - B1. Thermal displacements against temperature
  - B2. Comparison with DFT and experiment at 300 K
  - B3. Anisotropic displacements of ZnO
  - B4. X-ray and neutron diffraction
  - B5. Electron diffraction
  - B6. q-point mesh
- Known limitations

## Background

The thermal displacement matrix $U$ of site $\kappa$ is a sum over the phonon
modes $q\nu$ of a q-point mesh with $N$ points:

$$U_{\alpha\beta}(\kappa) = \frac{\hbar}{2 N m_\kappa} \sum_{q\nu}
\frac{1 + 2 n_{q\nu}}{\omega_{q\nu}} e_\alpha(\kappa, q\nu) e^*_\beta(\kappa, q\nu)$$

where $m_\kappa$ is the mass of the site, $\omega_{q\nu}$ and $e(\kappa, q\nu)$ are
the frequency and the eigenvector of the mode, and $n_{q\nu}$ is its Bose-Einstein
occupation. The Debye-Waller factor of the site for the reciprocal lattice vector
$g$, with $|g| = 1/d_{hkl}$, is $\exp(-2\pi^2 g^T U g)$. For an isotropic
$U = U_\mathrm{iso} I$, this is $\exp(-B s^2)$ with $B = 8\pi^2 U_\mathrm{iso}$
and $s = 1/(2 d_{hkl})$.

**Step 1: Relaxation.** The structure is relaxed with the force field.

**Step 2: Phonons.** The phonon workflow computes the force constants from
displaced supercells, and for the polar materials the Born charges and the
dielectric tensor. It then computes $U$ of each site of the primitive cell from
0 to 1000 K in steps of 100 K. The q-point mesh has 30000 q-points per reciprocal
atom, rounded up to odd mesh numbers. The three acoustic modes at Gamma are left
out.

**Step 3: Diffraction patterns.** pymatgen computes the X-ray (XRD), neutron (ND)
and electron (TEM) diffraction patterns with the Debye-Waller factors at each
temperature, and once without them.

The same workflow runs with VASP as `atomate2.vasp.flows.debye_waller.DebyeWallerMaker`.

## Installation

```
pip install 'atomate2[phonons,ase]' mp-api
pip install git+https://github.com/mdi-group/mace-field.git@45d5c5fa7b40a155855b3d155df1760e36849e64
pip install git+https://github.com/hrushikesh-s/pymatgen.git@c6de1930577320a8ba12a8910893e2343d9e8465
```

MACE-Field is a fork of MACE that installs as `mace-torch` and replaces MACE. It
also runs MACE-OMAT-0-medium. The anisotropic Debye-Waller factors are not in a
pymatgen release yet, so pymatgen installs from git. Part B needs only atomate2.

## Part A. Run the workflows

Part A needs the installation above, the model file below and a Materials Project
API key.

### Potentials

`mace_mp(model="medium-omat-0")` downloads and caches MACE-OMAT-0-medium, which is
released under the Academic Software License. For the Born charges, download
`MACEField-MH-0-omat-dielectric.model` from the
[MACE-Field releases](https://github.com/mdi-group/mace-field/releases).

Set the path to the downloaded file in the next cell. Set `DEVICE` to `"cpu"`
without a GPU. We use `float64`, since the force constants come from small force
differences.

In [ ]:
import os
import warnings

# macOS: conda's llvm-openmp and torch's bundled libomp both load and the
# duplicate aborts the process. This must be set before torch is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
warnings.filterwarnings("ignore")

# Replace the path with the file you downloaded.
MACE_FIELD_MODEL = "/path/to/MACEField-MH-0-omat-dielectric.model"
DEVICE = "cuda"
CALC_KWARGS = {"model": "medium-omat-0", "device": DEVICE, "default_dtype": "float64"}

### A1. Si step by step

We run Si with MACE-OMAT-0-medium. Part B shows our results.

#### Structure

We take Si (mp-149) from the Materials Project. Set `MP_API_KEY` first.

In [ ]:
from mp_api.client import MPRester

if not os.environ.get("MP_API_KEY"):
    raise OSError("export MP_API_KEY before running this cell")

with MPRester() as mpr:
    structure = mpr.get_structure_by_material_id("mp-149")
structure

#### Workflow

`DebyeWallerMaker.from_force_field_name` sets one force field for the relaxation
and the phonons. The model name in `calculator_kwargs` selects the MACE weights.
`xrd_kwargs` and `nd_kwargs` go to pymatgen's `XRDCalculator` and `NDCalculator`.
We use Mo K$\alpha$ X-rays and 1 Å neutrons. These shorter wavelengths give more
peaks up to $2\theta = 90°$ than the defaults. All other settings are the
defaults.

In [ ]:
from atomate2.forcefields.flows.debye_waller import DebyeWallerMaker

maker = DebyeWallerMaker.from_force_field_name(
    "MACE-MP-0",
    calculator_kwargs=CALC_KWARGS,
    xrd_kwargs={"wavelength": "MoKa"},
    nd_kwargs={"wavelength": 1.0},
)
flow = maker.make(structure)
flow.draw_graph().show()

#### Run

The run took 4 min on one A100 GPU of Perlmutter.

In [ ]:
from jobflow import JobStore, run_locally
from maggma.stores import MemoryStore

job_store = JobStore(MemoryStore(), additional_stores={"data": MemoryStore()})
responses = run_locally(flow, store=job_store, create_folders=True, ensure_success=True)
doc = responses[flow.output.uuid][1].output

#### Output

The output is a `DebyeWallerDocument`. It holds $U$ of each site at each
temperature, the primitive cell, and the XRD, ND and TEM patterns. The next cell
prints $U_\mathrm{iso}$ at 300 K and the number of peaks and spots.

In [ ]:
import numpy as np

data = doc.thermal_displacement_data
temperatures = np.array(data.temperatures_thermal_displacements)
u = np.array(data.thermal_displacement_matrix)
i300 = int(np.argmin(np.abs(temperatures - 300)))
{
    "temperatures in K": temperatures.tolist(),
    "U_iso(300 K) in A^2": (np.trace(u[i300], axis1=1, axis2=2) / 3).round(6).tolist(),
    "XRD peaks": len(doc.xrd_pattern_static.x),
    "ND peaks": len(doc.nd_pattern_static.x),
    "TEM spots": len(doc.tem_pattern_static),
}

### A2. Six materials and a q-point scan for Cu

`run_dw` builds and runs one workflow in its own folder. The loop after it runs
Cu, W, Si, NaCl, MgO and ZnO. NaCl, MgO and ZnO get the Born charges and the
dielectric tensor from MACE-Field. The loop also runs Cu at five more q-point
densities for B6. We ran these 11 runs at once, one per A100 GPU, in 3 to 4 min
each. Set `RUN_ALL = True` to run them.

In [ ]:
from pymatgen.core import Structure

from atomate2.common.schemas.debye_waller import DebyeWallerDocument
from atomate2.forcefields.jobs import ForceFieldDielectricMaker

# name: Materials Project id, and whether the NAC is used
MATERIALS = {
    "Cu": ("mp-30", False),
    "W": ("mp-91", False),
    "Si": ("mp-149", False),
    "NaCl": ("mp-22862", True),
    "MgO": ("mp-1265", True),
    "ZnO": ("mp-2133", True),
}


def run_dw(
    structure: Structure, polar: bool, kppa: int, root_dir: str
) -> DebyeWallerDocument:
    """Run the Debye-Waller workflow and return its output document.

    kppa is the q-point density of the thermal displacements. polar adds the
    MACE-Field Born charges and dielectric tensor for the NAC.
    """
    maker = DebyeWallerMaker.from_force_field_name(
        "MACE-MP-0",
        calculator_kwargs=CALC_KWARGS,
        xrd_kwargs={"wavelength": "MoKa"},
        nd_kwargs={"wavelength": 1.0},
    )
    if polar:
        maker.phonon_maker.born_maker = ForceFieldDielectricMaker(
            calculator_kwargs={"model": MACE_FIELD_MODEL, "device": DEVICE}
        )
    maker.phonon_maker.generate_frequencies_eigenvectors_kwargs = {
        "kpoint_density_thermal_displacements": kppa
    }
    flow = maker.make(structure)
    os.makedirs(root_dir, exist_ok=True)
    responses = run_locally(
        flow,
        store=JobStore(MemoryStore(), additional_stores={"data": MemoryStore()}),
        create_folders=True,
        root_dir=root_dir,
        ensure_success=True,
    )
    return responses[flow.output.uuid][1].output

In [ ]:
RUN_ALL = False
if RUN_ALL:
    with MPRester() as mpr:
        structures = {
            name: mpr.get_structure_by_material_id(mp_id)
            for name, (mp_id, _) in MATERIALS.items()
        }
    dw_docs = {
        name: run_dw(structures[name], polar, 30000, f"runs/{name}_30000")
        for name, (_, polar) in MATERIALS.items()
    }
    cu_scan = {
        kppa: run_dw(
            structures["Cu"], polar=False, kppa=kppa, root_dir=f"runs/Cu_{kppa}"
        )
        for kppa in (3000, 7000, 15000, 60000, 120000)
    }
    cu_scan[30000] = dw_docs["Cu"]

### A3. Reference from the Materials Project

The next cell computes $U$ from the force constants of the Materials Project's
Harmonic Phonon Database. These were computed with PBEsol and fitted with pheasy.
The cell uses the functions of the phonon workflow, so the NAC, the q-point mesh
and the temperatures are set as in A2.

In [ ]:
from phonopy import Phonopy
from pymatgen.io.phonopy import get_phonopy_structure

from atomate2.common.schemas.phonons import (
    _get_thermal_displacement_data,
    _set_nac_params,
)


def mp_reference(mpr: MPRester, mp_id: str) -> dict:
    """Return U from the pheasy force constants of the Materials Project."""
    summary = mpr.materials.summary.search(material_ids=[mp_id], fields=["phonon_IDs"])
    phonon_id = str(summary[0].phonon_IDs["pheasy"][0])
    mp_doc = mpr.materials.phonon.search(identifiers=[phonon_id], num_chunks=1)[0]
    phonon = Phonopy(
        get_phonopy_structure(mp_doc.structure),
        supercell_matrix=mp_doc.supercell_matrix,
        primitive_matrix=mp_doc.primitive_matrix,
    )
    phonon.force_constants = np.asarray(
        mpr.materials.phonon.get_forceconstants_from_phonon_id(phonon_id, "pheasy")
    )
    _set_nac_params(phonon, mp_doc.born, mp_doc.epsilon_static, 1e-4, "vasp")
    return {
        "phonon_id": phonon_id,
        "thermal_displacement_data": _get_thermal_displacement_data(phonon),
    }


if RUN_ALL:
    with MPRester() as mpr:
        mp_ref = {
            name: mp_reference(mpr, mp_id) for name, (mp_id, _) in MATERIALS.items()
        }

### A4. Saving the numbers for Part B

The next cell writes the numbers of A2 and A3 to
`tests/test_data/forcefields/debye_waller_tutorial/dw_results.json.gz`, which
Part B reads. It keeps the TEM spots of Si with at least 1% of the intensity of
the strongest spot. To plot your own runs, write them to another file and set
`RESULTS_FILE` in Part B.

In [ ]:
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path

from monty.serialization import dumpfn
from pymatgen.io.vasp import Kpoints

RESULTS_FILE = "../tests/test_data/forcefields/debye_waller_tutorial/dw_results.json.gz"


def sig(values: object, digits: int = 6) -> list[float]:
    """Round to a number of significant digits, to keep the file small."""
    return [float(f"{v:.{digits}g}") for v in np.ravel(values)]


def pattern_data(static: object, patterns: list) -> dict:
    """Return the peaks of a pattern without the factors and at each temperature."""
    return {
        "two_theta": sig(static.x),
        "d_hkl": sig(static.d_hkls),
        "hkl": [list(peak[0]["hkl"]) for peak in static.hkls],
        "intensity_static": sig(static.y),
        "intensity": [sig(pattern.y) for pattern in patterns],
    }


def u_data(data: dict) -> dict:
    """Return the Cartesian and CIF matrices U in A^2."""
    return {
        "U": np.round(data["thermal_displacement_matrix"], 6).tolist(),
        "Ucif": np.round(data["thermal_displacement_matrix_cif"], 6).tolist(),
    }


def export_results(dw_docs: dict, mp_ref: dict, cu_scan: dict, path: str) -> None:
    """Write the numbers that Part B plots."""
    materials = {
        name: {
            "material_id": MATERIALS[name][0],
            "structure": dw_doc.structure,
            "mace": u_data(dw_doc.thermal_displacement_data.model_dump()),
            "mp": u_data(mp_ref[name]["thermal_displacement_data"]),
            "xrd": pattern_data(dw_doc.xrd_pattern_static, dw_doc.xrd_patterns),
            "nd": pattern_data(dw_doc.nd_pattern_static, dw_doc.nd_patterns),
        }
        for name, dw_doc in dw_docs.items()
    }
    # TEM spots of Si with at least 1% of the intensity of the strongest spot
    si = dw_docs["Si"]
    spots = si.tem_pattern_static
    keep = [i for i, spot in enumerate(spots) if spot["intensity"] >= 0.01]
    tem = {
        "position": [spots[i]["position"] for i in keep],
        "hkl": [spots[i]["hkl"] for i in keep],
        "intensity_static": sig([spots[i]["intensity"] for i in keep]),
        "intensity": [sig([p[i]["intensity"] for i in keep]) for p in si.tem_patterns],
    }
    scan = {"kppa": [], "mesh": [], "U_iso": []}
    for kppa, dw_doc in sorted(cu_scan.items()):
        mesh = Kpoints.automatic_density(dw_doc.structure, kppa, force_gamma=True)
        u = np.array(dw_doc.thermal_displacement_data.thermal_displacement_matrix)
        scan["kppa"].append(kppa)
        # rounded up to odd numbers, as in the phonon flow
        scan["mesh"].append([int(n + 1 - n % 2) for n in mesh.kpts[0]])
        scan["U_iso"].append(sig(np.trace(u[:, 0], axis1=1, axis2=2) / 3))
    provenance = {
        "settings": "DebyeWallerMaker defaults with MACE-OMAT-0-medium in float64, "
        "xrd_kwargs {'wavelength': 'MoKa'} and nd_kwargs {'wavelength': 1.0}",
        "born_charges": f"{Path(MACE_FIELD_MODEL).name} for NaCl, MgO and ZnO",
        "reference": "pheasy force constants of the Materials Project's Harmonic "
        "Phonon Database",
        "versions": {
            package: version(package)
            for package in ("atomate2", "phonopy", "pymatgen", "mace-torch")
        },
        "date": datetime.now(UTC).date().isoformat(),
    }
    data = si.thermal_displacement_data
    dumpfn(
        {
            "provenance": provenance,
            "temperatures": data.temperatures_thermal_displacements,
            "materials": materials,
            "tem_si": tem,
            "cu_scan": scan,
        },
        path,
    )


if RUN_ALL:
    export_results(dw_docs, mp_ref, cu_scan, RESULTS_FILE)

## Part B. Our results

Part B draws all figures and tables from the numbers of our runs. It needs only
atomate2 and runs in seconds. The next cell loads the numbers and prints their
settings and package versions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from monty.serialization import loadfn

RESULTS_FILE = "../tests/test_data/forcefields/debye_waller_tutorial/dw_results.json.gz"
results = loadfn(RESULTS_FILE)
temperatures = np.array(results["temperatures"])
materials = results["materials"]
i300 = int(np.argmin(np.abs(temperatures - 300)))
results["provenance"]

### B1. Thermal displacements against temperature

The next cell plots $U_\mathrm{iso}$, one third of the trace of $U$, of each
element against temperature. Solid lines are MACE-OMAT-0-medium, dashed lines the
reference from the Materials Project (MP DFT).

In [ ]:
def u_iso(entry: dict, source: str) -> dict[str, np.ndarray]:
    """Return U_iso in A^2 at each temperature for each element of a material.

    source is "mace" or "mp". The sites of one element are averaged.
    """
    species = [site.specie.symbol for site in entry["structure"]]
    iso = np.trace(np.array(entry[source]["U"]), axis1=2, axis2=3) / 3
    result = {}
    for element in dict.fromkeys(species):
        sites = [i for i, s in enumerate(species) if s == element]
        result[element] = iso[:, sites].mean(axis=1)
    return result


fig, axes = plt.subplots(2, 3, figsize=(12, 6.5), sharex=True)
for ax, (name, entry) in zip(axes.flat, materials.items(), strict=True):
    mace, mp = u_iso(entry, "mace"), u_iso(entry, "mp")
    for i, element in enumerate(mace):
        ax.plot(temperatures, mace[element], color=f"C{i}", label=f"{element}, MACE")
        ax.plot(
            temperatures,
            mp[element],
            color=f"C{i}",
            ls="--",
            label=f"{element}, MP DFT",
        )
    ax.set_title(name)
    ax.legend(fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("Temperature (K)")
for ax in axes[:, 0]:
    ax.set_ylabel("$U_\\mathrm{iso}$ (Å$^2$)")
fig.tight_layout()
plt.show()

At 0 K, $U$ is the zero-point motion. At high temperature $U$ grows linearly with
temperature. From 300 to 1000 K, $U_\mathrm{iso}$ grows by a factor of 2.8 to 3.3,
close to the classical ratio of 10/3.

### B2. Comparison with DFT and experiment at 300 K

We compare $U_\mathrm{iso}$ at 300 K with these measurements:

- Si: neutron Pendellösung at 295.5 K, $B$ = 0.4761(17) Å² (Heacock et al.,
  [Science 373 (2021)](https://doi.org/10.1126/science.abc2794)).
- MgO: X-ray, $B$ = 0.30(1) Å² for Mg and 0.34(2) Å² for O (Lawrence,
  [Acta Cryst. A29, 94 (1973)](https://doi.org/10.1107/S0567739473000240)).
- ZnO: single-crystal X-ray at 293 K (Sowa and Ahsbahs,
  [J. Appl. Cryst. 39, 169 (2006)](https://doi.org/10.1107/S0021889805042457)).
  We use $U_\mathrm{iso} = (2 U_{11} + U_{33})/3$.
- NaCl: single-crystal X-ray at 295 K, Crystallography Open Database entry
  7132177 (Mettler et al., Chem. Commun. (2023)).
- Cu and W: $B$ at 293 K computed from measured phonon densities of states (Peng
  et al., Acta Cryst. A52, 456 (1996)). These are not direct measurements of $B$.

In [ ]:
# U_iso in A^2 at 293 to 295.5 K, from the references above
EXPERIMENT = {
    ("Cu", "Cu"): 0.5505 / (8 * np.pi**2),
    ("W", "W"): 0.1596 / (8 * np.pi**2),
    ("Si", "Si"): 0.4761 / (8 * np.pi**2),
    ("NaCl", "Na"): 0.0231,
    ("NaCl", "Cl"): 0.0204,
    ("MgO", "Mg"): 0.30 / (8 * np.pi**2),
    ("MgO", "O"): 0.34 / (8 * np.pi**2),
    ("ZnO", "Zn"): (2 * 0.0068 + 0.0061) / 3,
    ("ZnO", "O"): (2 * 0.007 + 0.008) / 3,
}

rows = {}
for (name, element), u_exp in EXPERIMENT.items():
    u_mace = u_iso(materials[name], "mace")[element][i300]
    u_mp = u_iso(materials[name], "mp")[element][i300]
    rows[(name, element)] = {
        "MACE": round(u_mace, 5),
        "MP DFT": round(u_mp, 5),
        "experiment": round(u_exp, 5),
        "MACE vs exp. (%)": round(100 * (u_mace / u_exp - 1), 1),
        "MP DFT vs exp. (%)": round(100 * (u_mp / u_exp - 1), 1),
    }
pd.DataFrame(rows).T

| Material | Element | MACE | MP DFT | Experiment | MACE vs exp. | MP DFT vs exp. |
|---|---|---|---|---|---|---|
| Cu | Cu | 0.00721 | 0.00684 | 0.00697 | +3.4% | −1.8% |
| W | W | 0.00219 | 0.00211 | 0.00202 | +8.5% | +4.2% |
| Si | Si | 0.00663 | 0.00685 | 0.00603 | +9.9% | +13.6% |
| NaCl | Na | 0.02312 | 0.02044 | 0.02310 | +0.1% | −11.5% |
| NaCl | Cl | 0.01891 | 0.01768 | 0.02040 | −7.3% | −13.3% |
| MgO | Mg | 0.00419 | 0.00393 | 0.00380 | +10.3% | +3.3% |
| MgO | O | 0.00420 | 0.00395 | 0.00431 | −2.4% | −8.2% |
| ZnO | Zn | 0.00759 | 0.00712 | 0.00657 | +15.5% | +8.4% |
| ZnO | O | 0.00707 | 0.00666 | 0.00733 | −3.6% | −9.2% |

$U_\mathrm{iso}$ at 300 K in Å².

MACE-OMAT-0-medium gives a 4% to 13% larger $U_\mathrm{iso}$ than the PBEsol
reference, except for Si, where it is 3% smaller. Both are within 16% of
experiment. The mean absolute difference from experiment is 6.8% for MACE and
8.2% for MP DFT. At the default q-point density, $U_\mathrm{iso}$ of Cu is 2.8%
below its converged value (B6).

### B3. Anisotropic displacements of ZnO

In wurtzite ZnO, $U$ has two independent components, $U_{11}$ along $a$ and
$U_{33}$ along $c$. The next cell compares them with the measurement of Sowa and
Ahsbahs at 293 K.

In [ ]:
# U11 and U33 in A^2 at 293 K (Sowa and Ahsbahs, 2006)
ZNO_EXPERIMENT = {"Zn": (0.0068, 0.0061), "O": (0.007, 0.008)}

species = [site.specie.symbol for site in materials["ZnO"]["structure"]]
rows = {}
for element, (u11_exp, u33_exp) in ZNO_EXPERIMENT.items():
    site = species.index(element)
    for source, label in (("mace", "MACE"), ("mp", "MP DFT")):
        u_cif = np.array(materials["ZnO"][source]["Ucif"])[i300, site]
        rows[(element, label)] = {"U11": u_cif[0, 0], "U33": u_cif[2, 2]}
    rows[(element, "experiment")] = {"U11": u11_exp, "U33": u33_exp}
anisotropy = pd.DataFrame(rows).T
anisotropy["U33/U11"] = anisotropy["U33"] / anisotropy["U11"]
anisotropy.round(4)

| Element | Source | $U_{11}$ | $U_{33}$ | $U_{33}/U_{11}$ |
|---|---|---|---|---|
| Zn | MACE | 0.0074 | 0.0079 | 1.06 |
| Zn | MP DFT | 0.0070 | 0.0074 | 1.06 |
| Zn | experiment | 0.0068(4) | 0.0061(4) | 0.90 |
| O | MACE | 0.0070 | 0.0073 | 1.04 |
| O | MP DFT | 0.0065 | 0.0069 | 1.05 |
| O | experiment | 0.007(2) | 0.008(2) | 1.14 |

$U$ at 300 K in Å².

Both calculations give $U_{33}$ larger than $U_{11}$ by 4% to 6% for both
elements. The measurement gives $U_{33}$ 10% smaller than $U_{11}$ for Zn. For O,
its uncertainty is larger than the anisotropy.

### B4. X-ray and neutron diffraction

The next cell plots the XRD pattern of Si without the Debye-Waller factors and at
300 and 1000 K. The three sets of peaks are shifted by 0.3° for clarity. The
intensities are not scaled.

In [ ]:
def plot_peaks(pattern: dict, title: str) -> None:
    """Plot the peaks without the factors and at 300 and 1000 K."""
    two_theta = np.array(pattern["two_theta"])
    _, ax = plt.subplots(figsize=(10, 3.5))
    for i, (intensity, label) in enumerate(
        (
            (pattern["intensity_static"], "without factors"),
            (pattern["intensity"][i300], "300 K"),
            (pattern["intensity"][-1], "1000 K"),
        )
    ):
        shift = 0.3 * (i - 1)
        ax.vlines(two_theta + shift, 0, intensity, label=label, color=f"C{i}")
    ax.set_xlabel("2$\\theta$ (°)")
    ax.set_ylabel("Intensity (not scaled)")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_peaks(materials["Si"]["xrd"], "Si, XRD with Mo K$\\alpha$")

For a material with one element, every peak gets the same factor $\exp(-2Bs^2)$
in its intensity. So $\ln(I_T/I_0) = -B/(2 d_{hkl}^2)$ is a straight line in
$1/d_{hkl}^2$, where $I_T$ and $I_0$ are the intensities at temperature $T$ and
without the factors. The next cell fits this line at 300 K for Cu, W and Si and
compares $B$ with $8\pi^2 U_\mathrm{iso}$.

In [ ]:
_, ax = plt.subplots(figsize=(6, 4))
rows = {}
for i, name in enumerate(("Cu", "W", "Si")):
    for kind, marker in (("xrd", "o"), ("nd", "x")):
        pattern = materials[name][kind]
        inv_d2 = 1 / np.array(pattern["d_hkl"]) ** 2
        intensity = np.array(pattern["intensity"][i300])
        log_ratio = np.log(intensity / np.array(pattern["intensity_static"]))
        slope = np.polyfit(inv_d2, log_ratio, 1)[0]
        rows[(name, kind.upper())] = {
            "B from the fit (A^2)": round(-2 * slope, 4),
            "8 pi^2 U_iso (A^2)": round(
                8 * np.pi**2 * u_iso(materials[name], "mace")[name][i300], 4
            ),
        }
        label = f"{name}, {kind.upper()}"
        ax.plot(inv_d2, log_ratio, marker, color=f"C{i}", label=label)
ax.set_xlabel("$1/d_{hkl}^2$ (Å$^{-2}$)")
ax.set_ylabel("$\\ln(I_{300\\,K}/I_0)$")
ax.legend()
plt.tight_layout()
plt.show()
pd.DataFrame(rows).T

| Material | XRD | ND | $8\pi^2 U_\mathrm{iso}$ |
|---|---|---|---|
| Cu | 0.5690 | 0.5690 | 0.5690 |
| W | 0.1731 | 0.1731 | 0.1731 |
| Si | 0.5232 | 0.5232 | 0.5232 |

$B$ at 300 K in Å², from the fit to the XRD and ND peaks.

The fits give back $8\pi^2 U_\mathrm{iso}$ to four digits for both XRD and ND.
In NaCl, MgO and ZnO, each element has its own factor. Their $\ln(I_T/I_0)$ is
then not a straight line in $1/d_{hkl}^2$.

### B5. Electron diffraction

`TEMCalculator` puts the beam along the third lattice vector of the primitive cell
by default. For Si this lattice vector has the length $a/\sqrt{2}$, where $a$ is
the cubic lattice parameter. It is therefore a ⟨110⟩ direction of the diamond
lattice. The next cell checks this length.

In [ ]:
si = materials["Si"]["structure"]
{
    "cubic a from the volume (A)": round((4 * si.volume) ** (1 / 3), 4),
    "sqrt(2) |a3| (A)": round(float(np.sqrt(2) * si.lattice.c), 4),
}

The next cell plots the spots without the Debye-Waller factors and at 1000 K, with
an area proportional to the intensity. Each pattern is normalized to its
strongest spot, the {111} spots.

In [ ]:
tem = results["tem_si"]
positions = np.array(tem["position"])
_, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for ax, intensity, title in (
    (axes[0], tem["intensity_static"], "Si, without factors"),
    (axes[1], tem["intensity"][-1], "Si, 1000 K"),
):
    ax.scatter(positions[:, 0], positions[:, 1], s=300 * np.array(intensity))
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlabel("x (cm)")
axes[0].set_ylabel("y (cm)")
plt.show()

The structure factor of each spot gets the factor $\exp(-2\pi^2 U_\mathrm{iso}/d^2)$,
and the intensity its square. After the normalization, the intensity of a spot at
temperature $T$ is its intensity without the factors times
$\exp(-4\pi^2 U_\mathrm{iso} (1/d^2 - 1/d_{111}^2))$. The next cell lists the
strongest spot of each family, with $h^2 + k^2 + l^2 = (a/d)^2$ in cubic indices,
and checks this relation at 1000 K.

In [ ]:
d_hkl = np.array([si.lattice.d_hkl(hkl) for hkl in tem["hkl"]])
static = np.array(tem["intensity_static"])
hot = np.array(tem["intensity"][-1])
u_1000 = u_iso(materials["Si"], "mace")["Si"][-1]
d_111 = d_hkl[np.argmax(static)]
expected = static * np.exp(-4 * np.pi**2 * u_1000 * (1 / d_hkl**2 - 1 / d_111**2))
a_cubic = (4 * si.volume) ** (1 / 3)

rows = {}
for i in np.argsort(-static):
    family = round((a_cubic / d_hkl[i]) ** 2)
    if family not in rows and len(rows) < 6:
        rows[family] = {
            "d (A)": round(d_hkl[i], 4),
            "without factors": round(static[i], 4),
            "1000 K": round(hot[i], 4),
            "expected at 1000 K": round(expected[i], 4),
        }
pd.DataFrame(rows).T.rename_axis("h^2+k^2+l^2")

| $h^2+k^2+l^2$ | $d$ (Å) | Without factors | 1000 K |
|---|---|---|---|
| 3 | 3.1314 | 1.0 | 1.0 |
| 8 | 1.9176 | 0.6622 | 0.5763 |
| 16 | 1.3559 | 0.2467 | 0.1719 |
| 11 | 1.6353 | 0.2137 | 0.1711 |
| 24 | 1.1071 | 0.1369 | 0.0764 |
| 19 | 1.2443 | 0.0957 | 0.0613 |

Normalized TEM intensities of Si.

The stored intensities agree with this relation within 0.02%. At 1000 K the {422}
spots lose 44% of their intensity relative to the {111} spots.

### B6. q-point mesh

The sum over the mesh leaves out the three acoustic modes at Gamma, so the cell
around Gamma is missing from the sum. $U$ is therefore too small, by an error that
falls as $1/N$ for an $N \times N \times N$ mesh. The next cell plots
$U_\mathrm{iso}$ of Cu at 300 K against $1/N$ for the six densities of A2. A
straight line through the two densest meshes gives the converged value $U_\infty$.

In [ ]:
scan = results["cu_scan"]
mesh = np.array([m[0] for m in scan["mesh"]])
u_300 = np.array([u[i300] for u in scan["U_iso"]])
slope, u_inf = np.polyfit(1 / mesh[-2:], u_300[-2:], 1)

_, ax = plt.subplots(figsize=(6, 4))
ax.plot(1 / mesh, u_300, "o", label="Cu, MACE")
x = np.linspace(0, 1 / mesh[0], 50)
ax.plot(x, u_inf + slope * x, "--", label="line through the two densest meshes")
ax.set_xlabel("$1/N$")
ax.set_ylabel("$U_\\mathrm{iso}$(300 K) (Å$^2$)")
ax.legend()
plt.show()

pd.DataFrame(
    {
        "q-points per reciprocal atom": scan["kppa"],
        "mesh": [f"{n}x{n}x{n}" for n in mesh],
        "U_iso (A^2)": u_300.round(6),
        "error (%)": (100 * (u_300 / u_inf - 1)).round(1),
    }
)

| q-points per reciprocal atom | Mesh | $U_\mathrm{iso}$ (Å²) | Error |
|---|---|---|---|
| 3000 | 15x15x15 | 0.006987 | −5.8% |
| 7000 | 19x19x19 | 0.007076 | −4.6% |
| 15000 | 25x25x25 | 0.007157 | −3.5% |
| 30000 | 31x31x31 | 0.007207 | −2.8% |
| 60000 | 39x39x39 | 0.007249 | −2.2% |
| 120000 | 49x49x49 | 0.007283 | −1.8% |

$U_\mathrm{iso}$ of Cu at 300 K, and its error against $U_\infty$ = 0.00741 Å².

The line through the two densest meshes predicts the four coarser meshes within
0.03%. The default density of 30000 gives $U_\mathrm{iso}$ 2.8% below $U_\infty$.
For a converged value, run two dense meshes and extrapolate in the same way.

## Known limitations

- $U$ is harmonic. The frequencies are those of the structure relaxed at 0 K, and
  are not renormalized with temperature. Thermal expansion and anharmonic effects
  are left out, and these grow with temperature.
- On a finite q-point mesh, $U$ is too small by an error that falls as $1/N$
  (B6).
- If the q-point mesh has imaginary modes, the phonon workflow stores no $U$, and
  the Debye-Waller workflow fails.
- The Born charges and the dielectric tensor come from MACE-Field.
- The references for Cu and W are computed from measured phonon densities of
  states. The reference for NaCl comes from a structure refinement, not from a
  dedicated measurement of the displacements.